<a href="https://colab.research.google.com/github/HarryKmK13/EECS183-Natural-Language-Processing-FA26/blob/main/Khant_KyawMin_hw4_c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW4 C: Text-to-speech
Implement a causal Transformer that predicts speech tokens from text. Frozen
CosyVoice2 audio models turn those tokens into speech. Use the full supplied
LibriTTS token cache. The default training budget is **one additional epoch**
from the provided two-epoch starter.

**43 points:** 16 for code, 24 for synthesized-speech word error rate (WER),
and 3 for report C1.
Code: dataset 2, collate 2, sequence construction 4, loss/causality 3, generation 2,
training 2, validation 1. Code credit is independent of model convergence.

| Synthesized-speech WER | Performance |
|---|---:|
| Below 15% | 24 |
| 15% to below 17% | 20 |
| 17% to below 19% | 16 |
| 19% to below 21% | 12 |
| 21% to below 22.5% | 8 |
| 22.5% to below 23.5% | 4 |
| 23.5% or higher | 0 |

The two-epoch starter measured 23.55% WER; the three-epoch staff reference reached
14.56%. Your results may differ. Missing predictions count as deletions.

**Report:** use the [template](https://github.com/Louis0324/eecs183-hw4/blob/main/report_template.txt)
for one selectable-text `report.pdf` covering **A1, A2, A3, B1, C1**. Put conceptual
answers in the PDF: aim for **3–5 sentences total per section**, plus requested
figures/sequences. Retain completed answers; mark unfinished items `Not completed`.


## Setup
Save a copy in Drive, select a Colab GPU, and run cells in order. Setup and data
downloads are automatic; no Hugging Face login is needed. If pip requests a
restart, restart and rerun setup.

Implement the TODOs using their docstrings. Keep the specified top-level
function/class names and signatures. Put extra imports and constants inside
functions/classes: the grader does not execute other top-level code.

Set `SMOKE_TEST=True` in setup for a quick check. For full experiments, set it to
`False`, choose a fresh `RESULTS` folder, and rerun from setup. Save your work to
Drive or download it before the runtime ends; Colab local storage is temporary.

| Implement | Provided |
|---|---|
| Dataset and collator | Cache loading and data loaders |
| Sequence construction, logits, loss, generation | Exact model constructor and starter loader |
| `train_epoch` and `validate` | Optimizer, scheduler, epoch loop, checkpoints/resume |
| Answer C1 | Audio synthesis, Whisper evaluation, submission export |

Keep the supplied constructor unchanged for checkpoint compatibility. In smoke
mode, check finite metrics, changing parameters during training, and unchanged
parameters during validation. In normal mode, evaluate all 200 test texts after
training, using your voice prompt.


In [1]:
REQUIREMENTS = 'requirements-tts.txt'
# Colab: select Runtime > Change runtime type > GPU before training.
import sys, subprocess
from pathlib import Path

# Course release settings are supplied by staff; no student login is needed.
COURSE_REPO_ID = 'Louis0324/eecs183-hw4'
COURSE_REVISION = 'hw4-2026-v4'
import os
os.environ.setdefault("HW4_HF_REPO", COURSE_REPO_ID)
os.environ.setdefault("HW4_HF_REVISION", COURSE_REVISION)

def ensure_hw4_support(repo_id=None, revision='hw4-2026-v4', packager_only=False):
    """Fetch the public support bundle, or use a local/uploaded offline copy."""
    import hashlib
    from io import BytesIO
    import json
    from pathlib import Path
    import re
    from urllib.parse import quote
    from urllib.request import urlopen
    import zipfile

    required = 'make_submission.py' if packager_only else 'course_download.py'
    if Path(required).is_file():
        return
    local_bundle = Path('hw4-student.zip')
    if local_bundle.is_file():
        content = local_bundle.read_bytes()
    elif repo_id:
        if not re.fullmatch(r'[\w.-]+/[\w.-]+', repo_id):
            raise ValueError('Expected a Hugging Face repository ID like COURSE_ORG/eecs183-hw4.')
        base = f'https://huggingface.co/datasets/{repo_id}/resolve/{quote(revision, safe="")}/'
        print('Downloading the small HW4 support bundle from Hugging Face; no login is required.')
        try:
            with urlopen(base + 'release_manifest.json', timeout=30) as response:
                manifest = json.loads(response.read(1024 * 1024))
            spec = manifest['files']['hw4-student.zip']
            if not 0 < spec['bytes'] < 30 * 1024 * 1024:
                raise ValueError('Invalid support bundle size in the release manifest.')
            with urlopen(base + 'hw4-student.zip', timeout=30) as response:
                content = response.read(spec['bytes'] + 1)
            if len(content) != spec['bytes'] or hashlib.sha256(content).hexdigest() != spec['sha256']:
                raise ValueError('Support bundle checksum mismatch; rerun to download a fresh copy.')
        except Exception as error:
            raise RuntimeError('Could not download the HW4 support release. Rerun this cell; '
                               'if needed, download hw4-student.zip from the course Hugging Face repo '
                               'and upload it through the Files sidebar before retrying.') from error
    else:
        # Allows local staff testing and offline/manual use before publication.
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError('Put hw4-student.zip in this folder or configure the course Hugging Face repository.') from error
        print('Upload hw4-student.zip to load the supplied helpers.')
        uploaded = files.upload()
        archives = [value for name, value in uploaded.items() if name.lower().endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('Upload exactly one hw4-student.zip support bundle, then rerun.')
        content = archives[0]

    with zipfile.ZipFile(BytesIO(content)) as archive:
        if archive.testzip() is not None:
            raise ValueError('Corrupt support ZIP; download a fresh copy.')
        for item in archive.infolist():
            path = Path(item.filename)
            if (path.is_absolute() or '..' in path.parts or
                    ((item.external_attr >> 16) & 0o170000) == 0o120000 or
                    not path.resolve().is_relative_to(Path.cwd().resolve())):
                raise ValueError('Unsafe support archive path.')
            # Browser/local notebook edits must never be replaced by starter copies.
            if item.is_dir() or path.suffix == '.ipynb' or (packager_only and item.filename != required):
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(item))
    if not Path(required).is_file():
        raise ValueError(f'This ZIP is missing {required}; use the current hw4-student.zip support bundle.')

ensure_hw4_support(COURSE_REPO_ID, COURSE_REVISION)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', REQUIREMENTS])
import torch
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('GPU:', 'Apple MPS' if torch.backends.mps.is_available() else (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'))
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
SMOKE_TEST = False  # True: only a few steps, for debugging your implementation.
DATA_ROOT = Path(os.environ.get('HW4_DATA_ROOT', 'data'))
RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
# Checkpoints/results disappear when Colab deletes a runtime. Copy them to Drive
# or download them before disconnecting. Submission ZIPs contain no checkpoints.

Python: 3.13.16 PyTorch: 2.14.1+cu130
GPU: Tesla T4


In [2]:
# Download only Part C data; verified cached copies are reused.
from course_download import ensure_course_data
DATA_ARCHIVE = None  # Optional offline override: Path("hw4-data-c.zip")
ensure_course_data('c', data_root=DATA_ROOT, archive=DATA_ARCHIVE)


Course file: hw4-data-c.zip (118.1 MiB); cached copies are reused.


hw4-data-c.zip:   0%|          | 0.00/124M [00:00<?, ?B/s]

Prepared data: data/tts


PosixPath('data/tts')

In [3]:
import json, math, copy
import numpy as np
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence, unpad_sequence
from sklearn.metrics import accuracy_score, r2_score
import matplotlib.pyplot as plt
from hw4_runtime import load_phoneme_data, load_mfcc_data, load_audio
np.random.seed(183); torch.manual_seed(183)
device = DEVICE

from IPython.display import Audio, display
class Config:
    IGNORE_ID = -1
    SOS_EOS_ID = 0  # Special-embedding index; NOT speech EOS.
    TASK_ID = 1    # Special-embedding index; NOT a speech-token offset.
    DATA_CACHE_DIR = str(DATA_ROOT / 'tts')
    PRETRAINED_DIR = os.environ.get('HW4_PRETRAINED_DIR', 'pretrained_models')
    RESULTS_DIR = str(RESULTS)

In [14]:
!git clone https://github.com/Louis0324/eecs183-hw4.git /content/eecs183-hw4

Cloning into '/content/eecs183-hw4'...
remote: Enumerating objects: 217, done.
remote: Counting objects: 100% (185/185), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 217 (delta 44), reused 176 (delta 40), pack-reused 32 (from 1)
Receiving objects: 100% (217/217), 694.39 KiB | 2.62 MiB/s, done.
Resolving deltas: 100% (51/51), done.


## 1. Load data and the text tokenizer
Training uses cached speech tokens. Audio models load later for synthesis to save
GPU memory. Use the text tokenizer's **full vocabulary**, including added tokens.


In [4]:
from hw4_util import download_pretrained_models, load_text_tokenizer
model_dir = download_pretrained_models(Config.PRETRAINED_DIR, tokenizer_only=True)
text_tokenizer = load_text_tokenizer(model_dir)
train_cache = torch.load(DATA_ROOT / 'tts/libritts_train.pt', map_location='cpu', weights_only=True)
val_cache = torch.load(DATA_ROOT / 'tts/libritts_dev.pt', map_location='cpu', weights_only=True)
train_data, val_data = train_cache['samples'], val_cache['samples']
speaker_to_idx = train_cache['metadata']['speaker_to_idx']
speech_vocab_size = train_cache['metadata']['speech_vocab_size']
print('Train:', len(train_data), 'Validation:', len(val_data), 'Speech vocabulary:', speech_vocab_size)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Text tokenizer loaded (Qwen2 BPE)
  Vocab size: 151663
Train: 354780 Validation: 9756 Speech vocabulary: 6561


## 2. Dataset and batching
Samples contain `utt_id`, `text`, `speaker_id`, and `speech_tokens`.

- Return long token tensors; reject empty/invalid samples or samples exceeding
  either limit (512 text tokens, 1,200 speech tokens) with `None`.
- Preserve text/audio pairing; do not truncate them independently. Unknown
  validation speakers use `speaker_idx=-1` (metadata only).
- The collator filters `None`, sorts all fields by decreasing speech length, and
  pads tokens with zero. Return `None` if all samples are invalid.

Use the full training cache outside smoke mode. Follow the return fields in the docstrings.


In [5]:
class CosyVoiceDataset(Dataset):

    def __init__(self, samples_list, text_tokenizer, speaker_to_idx_dict, max_text_len=512, max_speech_len=1200):
        self.samples = samples_list
        self.text_tokenizer = text_tokenizer
        self.speaker_to_idx = speaker_to_idx_dict
        self.max_text_len = max_text_len
        self.max_speech_len = max_speech_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        """Return dict: utt, text, text_token (long [T]), speech_token (long [S]), speaker_idx, speaker_id. Follow the validation and maximum-length rules above. Preserve the audio/text pairing."""
        sample = self.samples[idx]
        text = sample.get('text')
        speech = sample.get('speech_tokens')
        if not isinstance(text, str) or not text.strip() or speech is None:
            return None
        text_ids = self.text_tokenizer.encode(text)
        text_token = torch.as_tensor(text_ids, dtype=torch.long).flatten()
        speech_token = torch.as_tensor(speech, dtype=torch.long).flatten()
        if (not 0 < len(text_token) <= self.max_text_len or
                not 0 < len(speech_token) <= self.max_speech_len):
            return None
        if (torch.any(text_token < 0) or
                torch.any(text_token >= self.text_tokenizer.vocab_size) or
                torch.any(speech_token < 0)):
            return None
        speaker_id = sample.get('speaker_id', '')
        return dict(utt=sample['utt_id'], text=text,
                    text_token=text_token, speech_token=speech_token,
                    speaker_idx=int(self.speaker_to_idx.get(speaker_id, -1)),
                    speaker_id=speaker_id)

In [6]:
def cosyvoice_collate_fn(batch):
    """Return None for an all-invalid batch; otherwise sort by speech length descending and return utts, text, text_tokens, text_lengths, speech_tokens, speech_lengths, speaker_indices. Pad token tensors with 0; return lengths and indices as int64. Reorder every field consistently."""
    batch = [sample for sample in batch if sample is not None]
    if not batch:
        return None
    batch.sort(key=lambda sample: sample['speech_token'].numel(), reverse=True)
    return {
        'utts': [sample['utt'] for sample in batch],
        'text': [sample['text'] for sample in batch],
        'text_tokens': pad_sequence([sample['text_token'] for sample in batch],
                                    batch_first=True, padding_value=0),
        'text_lengths': torch.tensor([len(sample['text_token']) for sample in batch], dtype=torch.long),
        'speech_tokens': pad_sequence([sample['speech_token'] for sample in batch],
                                      batch_first=True, padding_value=0),
        'speech_lengths': torch.tensor([len(sample['speech_token']) for sample in batch], dtype=torch.long),
        'speaker_indices': torch.tensor([sample['speaker_idx'] for sample in batch], dtype=torch.long)
    }

In [7]:
train_dataset = CosyVoiceDataset(
    train_data[:5000],
    text_tokenizer,
    speaker_to_idx
)

val_dataset = CosyVoiceDataset(
    val_data,
    text_tokenizer,
    speaker_to_idx
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=cosyvoice_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=cosyvoice_collate_fn
)

first_batch = next(x for x in train_loader if x is not None)

print({
    k: tuple(v.shape)
    for k, v in first_batch.items()
    if torch.is_tensor(v)
})

{'text_tokens': (1, 10), 'text_lengths': (1,), 'speech_tokens': (1, 50), 'speech_lengths': (1,), 'speaker_indices': (1,)}


## 3. Causal speech-token LM
Use separate text, speech, and special-token embeddings. Speech IDs are
`0..speech_vocab_size-1`; speech **EOS = speech_vocab_size**. In the separate
special-token table, **SOS = 0** and **TASK = 1**.

For text `[t0,t1]` and speech `[s0,s1,s2]`:

```text
Inputs:  [SOS,t0,t1,TASK,s0,s1,s2]
Targets: [-1, -1, -1, s0,s1,s2,EOS]
```

TASK predicts the first speech token; the last speech input predicts EOS. Ignore
targets at SOS, text, and padding positions (`IGNORE_ID=-1`). The causal mask
blocks future positions only, allowing the diagonal; the padding mask blocks
padded positions. `True` means blocked. Compute loss/accuracy on non-ignored targets.
`_logits` adds positions and returns raw `[B,T,speech_vocab_size+1]` logits.

Keep the provided constructor/module names: **768 dimensions, 12 heads,
12 layers, 2,048 positions**. Honor dimension arguments for the grader's small
CPU checks. Smoke mode reduces data/steps, not the architecture.


In [8]:
class TextToSpeechLM(nn.Module):

    def __init__(self, text_vocab_size, speech_vocab_size, d_model=768, n_heads=12, n_layers=12, max_seq_len=2048):
        super().__init__()
        self.model_config = dict(d_model=d_model, n_heads=n_heads, n_layers=n_layers, max_seq_len=max_seq_len)
        self.d_model = d_model
        self.speech_vocab_size = speech_vocab_size
        self.text_embedding = nn.Embedding(text_vocab_size, d_model)
        self.speech_embedding = nn.Embedding(speech_vocab_size + 1, d_model)
        self.special_embedding = nn.Embedding(2, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4, dropout=0.0, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.output_projection = nn.Linear(d_model, speech_vocab_size + 1)

    def prepare_sequence(self, text_tokens, text_lengths, speech_tokens=None, speech_lengths=None):
        """Inputs: padded text/speech IDs and valid lengths. Return input embeddings [B,L,D], int64 targets [B,L], bool padding mask [B,L]. Follow the exact shift described above. If speech is None, return the text prefix and all ignored targets. Preserve embedding gradients."""
        from torch.nn.utils.rnn import pad_sequence
        device = text_tokens.device
        batch_size = text_tokens.shape[0]
        if speech_tokens is not None and speech_lengths is None:
            raise ValueError('speech_lengths required when speech_tokens is provided')
        sequences, targets = [], []
        for b in range(batch_size):
            tl = int(text_lengths[b].item())
            text_emb = self.text_embedding(text_tokens[b, :tl])
            sos = self.special_embedding(torch.tensor([0], device=device))
            task = self.special_embedding(torch.tensor([1], device=device))
            pieces = [sos, text_emb, task]
            labels = torch.full((tl + 2,), -1, dtype=torch.long, device=device)
            if speech_tokens is not None:
                sl = int(speech_lengths[b].item())
                speech_ids = speech_tokens[b, :sl]
                if sl:
                    pieces.append(self.speech_embedding(speech_ids))
                # TASK predicts the first speech token; last speech predicts EOS.
                labels = torch.cat((
                    torch.full((tl + 1,), -1, dtype=torch.long, device=device),
                    speech_ids,
                    torch.tensor([self.speech_vocab_size], dtype=torch.long, device=device)
                ))
            sequences.append(torch.cat(pieces, dim=0))
            targets.append(labels)
        embeddings = pad_sequence(sequences, batch_first=True, padding_value=0.0)
        target_ids = pad_sequence(targets, batch_first=True, padding_value=-1)
        lengths = torch.tensor([len(x) for x in sequences], device=device)
        padding = torch.arange(embeddings.shape[1], device=device).unsqueeze(0) >= lengths.unsqueeze(1)
        return embeddings, target_ids, padding


    def _logits(self, inputs, padding=None):
        """Add learned position embeddings. Apply a boolean strictly upper-triangular causal mask (diagonal=1) and optional boolean padding mask to the Transformer. Return raw logits [B,L,V+1]. Reject sequences longer than max_seq_len."""
        length = inputs.shape[1]
        if length > self.model_config['max_seq_len']:
            raise ValueError('Sequence exceeds max_seq_len')
        pos = torch.arange(length, device=inputs.device)
        x = inputs + self.position_embedding(pos).unsqueeze(0)
        # True blocks future positions, while diagonal and past stay visible.
        causal = torch.ones((length, length), dtype=torch.bool, device=inputs.device).triu(1)
        encoded = self.transformer(x, mask=causal, src_key_padding_mask=padding)
        return self.output_projection(encoded)

    def forward(self, text_tokens, text_lengths, speech_tokens, speech_lengths):
        """Call prepare_sequence and _logits; compute scalar cross entropy with ignore_index=-1 and scalar accuracy on non-ignored targets. Return (loss, accuracy)."""
        embeddings, targets, padding = self.prepare_sequence(
            text_tokens, text_lengths, speech_tokens, speech_lengths
        )
        logits = self._logits(embeddings, padding)
        valid = targets.ne(-1)
        if not valid.any():
            return logits.sum() * 0.0, logits.new_zeros(())
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]),
                               targets.reshape(-1), ignore_index=-1)
        accuracy = (logits.argmax(dim=-1)[valid] == targets[valid]).float().mean()
        return loss, accuracy

    @torch.no_grad()
    def generate(self, text_tokens, max_length=500, temperature=1.0, top_k=50, prompt_speech_token=None, prompt_text_tokens=None, min_length=None):
        """Batch size 1. Set eval mode and disable gradients. Prefix: SOS, prompt text (if any), target text, TASK, prompt speech (if any). Repeatedly call _logits, sample using temperature>0 and top-k (clamp k to vocabulary size; 0 disables top-k). Block EOS before min_length; stop at EOS or max_length new tokens. Return long [1,N] containing ONLY new speech tokens, excluding prompt and EOS. EOS ID is speech_vocab_size."""
        if text_tokens.ndim != 2 or text_tokens.shape[0] != 1:
            raise ValueError('generate expects text_tokens shaped [1, T]')
        if temperature <= 0:
            raise ValueError('temperature must be positive')
        self.eval()
        device = text_tokens.device
        embed_parts = [self.special_embedding(torch.tensor([0], device=device))]
        if prompt_text_tokens is not None and prompt_text_tokens.numel():
            embed_parts.append(self.text_embedding(prompt_text_tokens.reshape(-1).to(device)))
        embed_parts.append(self.text_embedding(text_tokens[0]))
        embed_parts.append(self.special_embedding(torch.tensor([1], device=device)))
        if prompt_speech_token is not None and prompt_speech_token.numel():
            embed_parts.append(self.speech_embedding(prompt_speech_token.reshape(-1).to(device).long()))
        prefix = torch.cat(embed_parts, dim=0).unsqueeze(0)
        produced = []
        eos = self.speech_vocab_size
        minimum = max(0, int(min_length)) if min_length is not None else 0
        for _ in range(int(max_length)):
            if prefix.shape[1] > self.model_config['max_seq_len']:
                break
            logits = self._logits(prefix)[:, -1, :] / temperature
            if len(produced) < minimum:
                logits[:, eos] = -float('inf')
            if top_k is not None and int(top_k) > 0:
                k = min(int(top_k), logits.shape[-1])
                threshold = torch.topk(logits, k, dim=-1).values[:, -1, None]
                logits = logits.masked_fill(logits < threshold, -float('inf'))
            sample = torch.multinomial(torch.softmax(logits, dim=-1), 1)
            token = int(sample.item())
            if token == eos:
                break
            produced.append(token)
            prefix = torch.cat((prefix, self.speech_embedding(sample)), dim=1)
        return torch.tensor([produced], dtype=torch.long, device=device)


In [9]:
# Keep the original architecture in smoke mode; only reduce the data/steps.
MODEL_KWARGS = dict(d_model=768, n_heads=12, n_layers=12, max_seq_len=2048)
model = TextToSpeechLM(text_tokenizer.vocab_size, speech_vocab_size, **MODEL_KWARGS).to(device)
print('Trainable parameters:', sum(p.numel() for p in model.parameters()))
# Use the course starter for normal training; smoke mode needs no checkpoint download.
# To resume after a saved epoch, restore RESULTS and set RESUME=True here.
RESUME = False
USE_STARTER = not SMOKE_TEST
STARTER_CHECKPOINT = os.environ.get('HW4_TTS_STARTER')  # Optional local file override.
starter_metadata = None
if USE_STARTER and not RESUME:
    from tts_checkpoint import load_tts_starter
    from course_download import download_course_file
    STARTER_CHECKPOINT = download_course_file("hw4-tts-starter.pt", local_path=STARTER_CHECKPOINT)
    starter_metadata = load_tts_starter(model, text_tokenizer, STARTER_CHECKPOINT)
    print('Loaded course starter; completed reference epochs:',
          starter_metadata['source_training']['completed_epochs'])


Trainable parameters: 213191842
Course file: hw4-tts-starter.pt (406.7 MiB); cached copies are reused.


hw4-tts-starter.pt:   0%|          | 0.00/426M [00:00<?, ?B/s]

Loaded course starter; completed reference epochs: 2


## 4. Training and validation
Implement the functions below:

- **Training:** skip `None` batches, move tensors to the device, use train mode,
  clear gradients, backpropagate, clip gradients, and step the optimizer and
  scheduler once per valid batch (`scheduler` may be `None`).
- **Validation:** use eval mode and no gradients; leave parameters unchanged.
- Return mean loss/accuracy over valid batches, or `(0.0, 0.0)` if none are valid.

The optimizer, scheduler, epoch loop, and checkpoint selection are provided.
Defaults: AdamW, learning rate `2e-4`, weight decay `0.01`, 10% warmup then cosine
decay. Tuning is optional; use validation results. Reduce batch size if needed.

**Warm start:** the model cell loads the two-epoch starter. The default is one
further epoch with a fresh optimizer. `num_epochs` is the total student budget;
it excludes the starter's two epochs. Smoke mode skips the starter; setting
`USE_STARTER=False` before the model-loading conditional trains from scratch.

Checkpoints save at epoch end; a runtime reset loses any unfinished epoch.
To resume, restore `tts_latest.pt` and `tts_best.pt` in `RESULTS`, set `RESUME=True`
in the model cell, and rerun from there with the same training settings.
Report the training you completed, even if you stopped early.


In [10]:
def train_epoch(model, dataloader, optimizer, scheduler, device):
    """Implement one epoch with the exact behavior above. Move text_tokens, text_lengths, speech_tokens, speech_lengths to device. scheduler may be None. Return (mean_loss, mean_accuracy)."""
    import contextlib, time
    device = torch.device(device)
    use_amp = device.type == 'cuda'          # fp16 tensor cores on GPU; plain fp32 on CPU
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    total = len(dataloader) if hasattr(dataloader, '__len__') else None
    start = time.time()
    model.train()
    loss_sum, acc_sum, n = 0.0, 0.0, 0
    for i, batch in enumerate(dataloader):
        if batch is None:
            continue
        text = batch['text_tokens'].to(device)
        text_lengths = batch['text_lengths'].to(device)
        speech = batch['speech_tokens'].to(device)
        speech_lengths = batch['speech_lengths'].to(device)
        optimizer.zero_grad(set_to_none=True)
        amp = torch.autocast('cuda', dtype=torch.float16) if use_amp else contextlib.nullcontext()
        with amp:
            loss, accuracy = model(text, text_lengths, speech, speech_lengths)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)            # clip the real (unscaled) gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        loss_sum += float(loss.detach())
        acc_sum += float(accuracy.detach())
        n += 1
        if n % 1000 == 0:
            msg = f'step {i+1}' + (f'/{total}' if total else '') + f'  loss {loss_sum/n:.4f}  acc {acc_sum/n:.2%}'
            if total:
                msg += f'  ~{(time.time()-start)/(i+1)*(total-i-1)/60:.0f} min left'
            print(msg, flush=True)
    if n == 0:
        return 0.0, 0.0
    return loss_sum / n, acc_sum / n

In [11]:
def validate(model, dataloader, device):
    """Evaluate valid batches with model.eval() and torch.no_grad(). No updates or gradients. Return (mean_loss, mean_accuracy), or (0.0,0.0) for no valid batches."""
    import contextlib
    device = torch.device(device)
    use_amp = device.type == 'cuda'
    model.eval()
    loss_sum, acc_sum, n = 0.0, 0.0, 0
    with torch.no_grad():
        for batch in dataloader:
            if batch is None:
                continue
            text = batch['text_tokens'].to(device)
            text_lengths = batch['text_lengths'].to(device)
            speech = batch['speech_tokens'].to(device)
            speech_lengths = batch['speech_lengths'].to(device)
            amp = torch.autocast('cuda', dtype=torch.float16) if use_amp else contextlib.nullcontext()
            with amp:
                loss, accuracy = model(text, text_lengths, speech, speech_lengths)
            loss_sum += float(loss)
            acc_sum += float(accuracy)
            n += 1
    if n == 0:
        return 0.0, 0.0
    return loss_sum / n, acc_sum / n

In [12]:
from hw4_util import get_warmup_cosine_scheduler, save_checkpoint
if USE_STARTER and not RESUME and starter_metadata is None:
    raise RuntimeError('Load the course starter successfully before training, or set USE_STARTER=False.')
num_epochs = 1 if (SMOKE_TEST or USE_STARTER) else 3
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
total_steps = num_epochs * len(train_loader)
scheduler = get_warmup_cosine_scheduler(optimizer, max(1, total_steps // 10), total_steps)
best_val_loss, start_epoch = float('inf'), 0
if RESUME:
    if not (RESULTS / 'tts_latest.pt').exists():
        raise FileNotFoundError('No tts_latest.pt in RESULTS; restore your saved checkpoint or set RESUME=False.')
    state = torch.load(RESULTS / 'tts_latest.pt', map_location='cpu', weights_only=True)
    if state.get('model_config') != MODEL_KWARGS:
        raise ValueError('Resume checkpoint architecture does not match the original TTS model.')
    model.load_state_dict(state['model_state_dict'], strict=True)
    starter_metadata = state.get('starter_metadata')
    optimizer.load_state_dict(state['optimizer_state_dict'])
    if state.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(state['scheduler_state_dict'])
    start_epoch = state['epoch'] + 1
    if (RESULTS / 'tts_best.pt').exists():
        best_val_loss = torch.load(RESULTS / 'tts_best.pt', map_location='cpu', weights_only=True)['val_loss']
for epoch in range(start_epoch, num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc = validate(model, val_loader, device)
    print(f'Epoch {epoch+1}: train={train_loss:.4f}, dev={val_loss:.4f}, dev accuracy={val_acc:.2%}')
    save_checkpoint(model, optimizer, scheduler, epoch, train_loss, val_loss, RESULTS / 'tts_latest.pt', model_config=MODEL_KWARGS, starter_metadata=starter_metadata)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, None, None, epoch, train_loss, val_loss, RESULTS / 'tts_best.pt', model_config=MODEL_KWARGS, starter_metadata=starter_metadata)


step 1000/5000  loss 4.7932  acc 9.86%  ~9 min left
step 2000/5000  loss 4.9748  acc 9.11%  ~7 min left
step 3000/5000  loss 4.9697  acc 9.15%  ~4 min left
step 4000/5000  loss 4.9062  acc 9.43%  ~2 min left
step 5000/5000  loss 4.8347  acc 9.75%  ~0 min left
Epoch 1: train=4.8347, dev=5.0246, dev accuracy=8.97%


KeyboardInterrupt: 

## 5. Check audio decoding
First decode ground-truth tokens to check the audio pipeline before testing your
LM. Finish and save training before running this cell: it downloads frozen
audio models and frees optimizer state to save memory.


In [16]:
# Checkpoints are saved. Release training state before loading the audio models.
import gc
model.zero_grad(set_to_none=True)
globals().pop('scheduler', None)
globals().pop('optimizer', None)
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()

from hw4_util import load_speech_tokenizer, load_flow_model, load_vocoder, generate_audio_from_tokens, synthesize
model_dir = download_pretrained_models(Config.PRETRAINED_DIR)
speech_tokenizer = load_speech_tokenizer(model_dir, device)
flow_model = load_flow_model(model_dir, device)
vocoder = load_vocoder(model_dir, device)
sample = next(x for x in (val_dataset[i] for i in range(len(val_dataset))) if x is not None)
audio, mel = generate_audio_from_tokens(sample['speech_token'][:50] if SMOKE_TEST else sample['speech_token'], flow_model, vocoder, device)
display(Audio(audio.cpu().numpy(), rate=24000))
# Use the supplied two-epoch pretrained starter model
# instead of a missing training checkpoint.
model.eval()
audio, sr = synthesize('This is a short speech synthesis test.', model, text_tokenizer, speech_tokenizer, flow_model, vocoder, device,
                       output_path=RESULTS / 'tts_example.wav', max_length=20 if SMOKE_TEST else 500)
display(Audio(audio.numpy(), rate=sr))

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

  - Checkpoint: flow.pt
  - Device: cuda
  - Checkpoint: hift.pt
  - Device: cuda


In [19]:
from google.colab import files
from pathlib import Path
import soundfile as sf
import shutil

# Upload the WAV recording
uploaded = files.upload()

# Create results folder
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

# Save with the filename expected by your notebook
uploaded_filename = next(iter(uploaded))
source = Path(uploaded_filename)
destination = RESULTS / "voice_prompt.wav"

shutil.copyfile(source, destination)

print("File exists:", destination.exists())
print("Saved to:", destination.resolve())
print("Audio information:", sf.info(str(destination)))

Saving voice_prompt_pcm16.wav to voice_prompt_pcm16.wav
File exists: True
Saved to: /content/results/voice_prompt.wav
Audio information: results/voice_prompt.wav
samplerate: 16000 Hz
channels: 1
duration: 5.032 s
format: WAV (Microsoft) [WAV]
subtype: Signed 16 bit PCM [PCM_16]


## 6. Voice prompt and evaluation
Use the best validation checkpoint. Record **3–10 seconds of your speech** as
WAV/FLAC. Read the sentence in `prompt_text`, or edit that argument to match your
recording **exactly**. Replace `YOUR_STUDENT_ID`, then run the cell and upload the
recording. The helper resamples to mono 16 kHz. For a saved file, pass
`audio_path=...` to `record_or_load_voice`.

Synthesize all **200 test texts**, transcribe them with Whisper, and save
`RESULTS/tts_submission.json` with the original text IDs. Submit Whisper's outputs
unchanged. Audio stays local for listening and is excluded from the ZIP.

Rerun to resume. New model weights or prompts start fresh predictions. After
changing decoding code/settings, use `RESTART_PREDICTIONS=True` once, then `False`
to resume interrupted evaluation.


In [20]:
from pathlib import Path
import soundfile as sf

voice_path = RESULTS / "voice_prompt.wav"

print("Working directory:", Path.cwd())
print("Recording path:", voice_path.resolve())
print("File exists:", voice_path.exists())

if voice_path.exists():
    print("File size:", voice_path.stat().st_size, "bytes")
    try:
        print("Audio information:", sf.info(str(voice_path)))
    except Exception as e:
        print("Audio is unreadable:", e)

Working directory: /content
Recording path: /content/results/voice_prompt.wav
File exists: True
File size: 161102 bytes
Audio information: results/voice_prompt.wav
samplerate: 16000 Hz
channels: 1
duration: 5.032 s
format: WAV (Microsoft) [WAV]
subtype: Signed 16 bit PCM [PCM_16]


In [21]:
from hw4_util import record_or_load_voice, run_voice_cloning_evaluation
prompt_audio, prompt_text, recording_path = record_or_load_voice(
    prompt_text='The quick brown fox jumps over the lazy dog.', results_dir=RESULTS, audio_path=RESULTS / 'voice_prompt.wav')
STUDENT_ID = '3040515312'  # Replace this; no ID-based shuffling is performed.
RESTART_PREDICTIONS = False  # True after changing generation code/settings.
submission_path = run_voice_cloning_evaluation(
    STUDENT_ID, model, text_tokenizer, speech_tokenizer, flow_model, vocoder,
    prompt_audio, prompt_text, results_dir=RESULTS, pretrained_dir=Config.PRETRAINED_DIR,
    device=device, model_dir=model_dir, max_examples=2 if SMOKE_TEST else None, restart=RESTART_PREDICTIONS)
print('Saved', submission_path)

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

... (6 outputs left out)

Device set to use cuda
`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.


Saved example 0; 1/200 attempted.
Saved example 1; 2/200 attempted.
Saved example 2; 3/200 attempted.
Saved example 3; 4/200 attempted.
Saved example 4; 5/200 attempted.
Saved example 5; 6/200 attempted.
Saved example 6; 7/200 attempted.
Saved example 7; 8/200 attempted.
Saved example 8; 9/200 attempted.
Saved example 9; 10/200 attempted.


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Saved example 10; 11/200 attempted.
Saved example 11; 12/200 attempted.
Saved example 12; 13/200 attempted.
Saved example 13; 14/200 attempted.
Saved example 14; 15/200 attempted.
Saved example 15; 16/200 attempted.
Saved example 16; 17/200 attempted.
Saved example 17; 18/200 attempted.
Saved example 18; 19/200 attempted.
Saved example 19; 20/200 attempted.
Saved example 20; 21/200 attempted.
Saved example 21; 22/200 attempted.
Saved example 22; 23/200 attempted.
Saved example 23; 24/200 attempted.
Saved example 24; 25/200 attempted.
Saved example 25; 26/200 attempted.
Saved example 26; 27/200 attempted.
Saved example 27; 28/200 attempted.
Saved example 28; 29/200 attempted.
Saved example 29; 30/200 attempted.
Saved example 30; 31/200 attempted.
Saved example 31; 32/200 attempted.
Saved example 32; 33/200 attempted.
Saved example 33; 34/200 attempted.
Saved example 34; 35/200 attempted.
Saved example 35; 36/200 attempted.
Saved example 36; 37/200 attempted.
Saved example 37; 38/200 att

## Report C1: Teacher forcing, masks, and WER (3 points)
Answer in **C1** of `report.pdf` (about 3–5 sentences total, plus sequences).
No extra training is required:

1. For text `[t0]` and speech `[s0,s1]`, write the input and target sequences,
   including SOS, TASK, EOS, and ignored targets. Which position predicts the
   first speech token? (1 point)
2. What leaks without a causal mask, and why exclude padded positions? (1 point)
3. Why does low synthesized-speech WER not establish similarity to the prompt
   speaker's voice? (1 point)


## Prepare your submission
The final cell packages your current notebook, available predictions/plots, and
**one shared `report.pdf`**. Checkpoints, datasets, and audio are excluded.
Combine separate reports into one PDF before uploading.

1. Leave `include_previous=True`. Upload `hw4-ab.zip` at the earlier-ZIP prompt
   (or select separate A and B ZIPs together). For C only, cancel this prompt.
2. Upload your final combined `report.pdf`, or cancel to keep the imported/saved
   PDF, if any.
3. Upload the downloaded **`submission.zip`** to Gradescope.

**Partial work is accepted:** completed questions are graded independently.
Check the printed file list and your PDF before uploading. Every Gradescope
resubmission must contain **all** work you want graded.

**Another order or runtime reset:** set `include_previous=True` and import your
latest ZIP in whichever notebook you package last. The final cell needs no
training/setup or GPU. To include results saved on Drive, mount Drive and set
`RESULTS` to that folder's path before packaging.

Saved predictions: `RESULTS/tts_submission.json` (200 test texts).


In [23]:
include_report = True  # Upload one combined PDF, or cancel to keep an imported/saved PDF.
include_previous = True  # Merge earlier submission ZIPs; cancel if none.

# Packaging needs no training dependencies, GPU, or data download.
# Course release settings are supplied by staff; no student login is needed.
COURSE_REPO_ID = 'Louis0324/eecs183-hw4'
COURSE_REVISION = 'hw4-2026-v4'
import os
os.environ.setdefault("HW4_HF_REPO", COURSE_REPO_ID)
os.environ.setdefault("HW4_HF_REVISION", COURSE_REVISION)

def ensure_hw4_support(repo_id=None, revision='hw4-2026-v4', packager_only=False):
    """Fetch the public support bundle, or use a local/uploaded offline copy."""
    import hashlib
    from io import BytesIO
    import json
    from pathlib import Path
    import re
    from urllib.parse import quote
    from urllib.request import urlopen
    import zipfile

    required = 'make_submission.py' if packager_only else 'course_download.py'
    if Path(required).is_file():
        return
    local_bundle = Path('hw4-student.zip')
    if local_bundle.is_file():
        content = local_bundle.read_bytes()
    elif repo_id:
        if not re.fullmatch(r'[\w.-]+/[\w.-]+', repo_id):
            raise ValueError('Expected a Hugging Face repository ID like COURSE_ORG/eecs183-hw4.')
        base = f'https://huggingface.co/datasets/{repo_id}/resolve/{quote(revision, safe="")}/'
        print('Downloading the small HW4 support bundle from Hugging Face; no login is required.')
        try:
            with urlopen(base + 'release_manifest.json', timeout=30) as response:
                manifest = json.loads(response.read(1024 * 1024))
            spec = manifest['files']['hw4-student.zip']
            if not 0 < spec['bytes'] < 30 * 1024 * 1024:
                raise ValueError('Invalid support bundle size in the release manifest.')
            with urlopen(base + 'hw4-student.zip', timeout=30) as response:
                content = response.read(spec['bytes'] + 1)
            if len(content) != spec['bytes'] or hashlib.sha256(content).hexdigest() != spec['sha256']:
                raise ValueError('Support bundle checksum mismatch; rerun to download a fresh copy.')
        except Exception as error:
            raise RuntimeError('Could not download the HW4 support release. Rerun this cell; '
                               'if needed, download hw4-student.zip from the course Hugging Face repo '
                               'and upload it through the Files sidebar before retrying.') from error
    else:
        # Allows local staff testing and offline/manual use before publication.
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError('Put hw4-student.zip in this folder or configure the course Hugging Face repository.') from error
        print('Upload hw4-student.zip to load the supplied helpers.')
        uploaded = files.upload()
        archives = [value for name, value in uploaded.items() if name.lower().endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('Upload exactly one hw4-student.zip support bundle, then rerun.')
        content = archives[0]

    with zipfile.ZipFile(BytesIO(content)) as archive:
        if archive.testzip() is not None:
            raise ValueError('Corrupt support ZIP; download a fresh copy.')
        for item in archive.infolist():
            path = Path(item.filename)
            if (path.is_absolute() or '..' in path.parts or
                    ((item.external_attr >> 16) & 0o170000) == 0o120000 or
                    not path.resolve().is_relative_to(Path.cwd().resolve())):
                raise ValueError('Unsafe support archive path.')
            # Browser/local notebook edits must never be replaced by starter copies.
            if item.is_dir() or path.suffix == '.ipynb' or (packager_only and item.filename != required):
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(item))
    if not Path(required).is_file():
        raise ValueError(f'This ZIP is missing {required}; use the current hw4-student.zip support bundle.')

ensure_hw4_support(COURSE_REPO_ID, COURSE_REVISION, packager_only=True)
try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None
from make_submission import build_submission, from_colab
if colab_files is not None:
    from_colab('c', include_report=include_report, include_previous=include_previous,
               results_dir=globals().get("RESULTS", "results"))
else:
    build_submission(report="report.pdf" if include_report else None)


Step 1: upload your latest combined submission ZIP from earlier parts.
If you have separate A/B/C ZIPs, select the ones with different parts together. Do not select old and new versions of the same part.
No earlier work to add? Cancel this upload. Do not upload the course support bundle or data archives here.


Saving submission(2).zip to submission(2) (1).zip
Imported submission(2) (1).zip: 35 files.
Report: upload ONE combined PDF with sections A1, A2, A3, B1, C1. Separate PDFs are not merged.
Cancel to keep the report already imported/saved, or continue without one if it is not ready.


Saving HW4_report_abc.pdf to HW4_report_abc.pdf
Using HW4_report_abc.pdf as the shared report.pdf.
Created submission.zip: 37 available files, 3.44 MiB compressed.
Included notebooks: A, B, C. Shared report.pdf: included.
   hw4-a.ipynb
   hw4-b.ipynb
   hw4-c.ipynb
   report.pdf
   results/a/mfcc_layer_0.json
   results/a/mfcc_layer_1.json
   results/a/mfcc_layer_10.json
   results/a/mfcc_layer_11.json
   results/a/mfcc_layer_12.json
   results/a/mfcc_layer_2.json
   results/a/mfcc_layer_3.json
   results/a/mfcc_layer_4.json
   results/a/mfcc_layer_5.json
   results/a/mfcc_layer_6.json
   results/a/mfcc_layer_7.json
   results/a/mfcc_layer_8.json
   results/a/mfcc_layer_9.json
   results/a/mfcc_r2.png
   results/a/mfcc_results.json
   results/a/phoneme_accuracy.png
   results/a/phoneme_layer_0.json
   results/a/phoneme_layer_1.json
   results/a/phoneme_layer_10.json
   results/a/phoneme_layer_11.json
   results/a/phoneme_layer_12.json
   results/a/phoneme_layer_2.json
   results/a/pho

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>